In [ ]:
######## Recadrage + Noir et Blanc ###############
import cv2
import os
import numpy as np

# -------------------------
# CONFIGURATION GLOBALE
# -------------------------

# Définir le dossier source (images d'entrée)
source_dir = r".\donnees\train"

# Définir le dossier de sortie (images traitées)
destination_dir = r".\donnees\resized2\train"

# Dimensions utilisées temporairement pour faciliter la détection de cercles
resize_width, resize_height = 480, 256



# -------------------------------------
# TRAITEMENT DE CHAQUE IMAGE
# -------------------------------------

# Parcourir récursivement tous les fichiers dans le dossier source
for root, _, files in os.walk(source_dir):
    for file in files:
        # Filtrer uniquement les fichiers image
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):  # Filtrer les images
            source_path = os.path.join(root, file)
            
            # Reconstituer le chemin de destination en conservant la structure du dossier source
            relative_path = os.path.relpath(source_path, source_dir)
            destination_path = os.path.join(destination_dir, relative_path)
            destination_folder = os.path.dirname(destination_path)
            os.makedirs(destination_folder, exist_ok=True)  # Créer le dossier de sortie si nécessaire

            # Charger l'image originale
            image0 = cv2.imread(source_path)
            if image0 is None:
                print(f"Erreur : Impossible de charger {source_path}")
                continue

            # Redimensionner temporairement à 480x256 pour accélérer la détection de cercle
            image = cv2.resize(image0, (resize_width, resize_height))

            # Convertir en niveaux de gris (nécéssaire pour HoughCircles)
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

            # Détecter les cercles à l’aide de la transformée de Hough
            circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=300, param1=100, param2=10, minRadius=100, maxRadius=200)

            # Obtenir le centre de l'image redimensionnée
            height, width = gray.shape
            center_x, center_y = width // 2, height // 2

            # Chercher le cercle détecté le plus proche du centre
            best_circle = None
            min_distance = float("inf")

            if circles is not None:
                for circle in circles[0, :]:
                    x, y, r = circle
                    distance_to_center = np.sqrt((x - center_x) ** 2 + (y - center_y) ** 2)
                    if distance_to_center < min_distance:
                        min_distance = distance_to_center
                        best_circle = circle

            
            # -----------------------------------------------
            # RECADRAGE SUR L’IMAGE ORIGINALE + NOIR ET BLANC
            # -----------------------------------------------
            if best_circle is not None:
                x, y, r = best_circle  # Coordonnées du cercle sur l'image redimensionnée
                
                # Mise à l'échelle des coordonnées et du rayon vers l’image originale
                scale_x = image0.shape[1] / resize_width
                scale_y = image0.shape[0] / resize_height
                x = int(x * scale_x)
                y = int(y * scale_y)
                r = int(r * min(scale_x, scale_y))  # Rayon ajusté à l’échelle moyenne

               # Calcul des coordonnées du carré à extraire autour du cercle
                x1, y1 = max(0, x - r), max(0, y - r)
                x2, y2 = min(image0.shape[1], x + r), min(image0.shape[0], y + r)

                # Recadrer l'image originale
                cropped_image = image0[y1:y2, x1:x2]

                #Convertir l’image recadrée en niveaux de gris (noir et blanc)
                cropped_gray = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2GRAY)

                # Enregistrer l’image traitée au bon emplacement
                cv2.imwrite(destination_path, cropped_gray)
                print(f"Image sauvegardée : {destination_path}")

print("🚀 Traitement terminé pour toutes les images !")